<a href="https://colab.research.google.com/github/netsetos/genai-engg-notebooks/blob/main/module-04-model-architecture/lesson-4.1-tokenization-and-embeddings/exercises-4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.1 - Tokenization & Embeddings

**Module 4 - Model Architecture** | Netsetos GenAI Engineering

This notebook builds the two steps every LLM takes before it can "think": **tokenization** (splitting text into pieces the model can look up) and **embeddings** (turning those pieces into meaning-carrying vectors). You'll build a BPE tokenizer from scratch, compare real tokenizers, price a prompt to the rupee, generate embeddings, and see the quirks (letter-counting, the multilingual token tax) that fall straight out of how tokenization works.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the three libraries this notebook uses and fix the random seed so every run is reproducible. Run this first. On Colab it installs into the session; run locally and it's a no-op if the packages already exist.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install tiktoken google-genai numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

This cell is pure environment prep - it installs the libraries and seeds randomness. There is no logic to trace; it just makes the rest of the notebook reproducible and importable.

**How the code works - step by step**
- `tiktoken` - OpenAI's fast tokenizer, used to inspect how GPT-4 / GPT-4o split text.
- `google-genai` - the current Gemini SDK, for the embedding calls further down.
- `numpy` - vector math for the embedding similarity matrix.
- `np.random.seed(42)` - pins NumPy's randomness so any seeded values are identical across runs.

**What you'll see:** (no output - environment prep)

## 1 - Build a BPE tokenizer from scratch

Byte-Pair Encoding is the algorithm behind GPT, Llama and most modern tokenizers, and it has exactly one idea: **find the most frequent adjacent pair of pieces, glue them into a single new piece, and repeat.** Start from raw characters; after a few merges, common strings like `we` or `lo` become single tokens while rare words stay split. Building it yourself is the fastest way to see why one word can be 1 token and another 5.

BPE repeats only three moves - **count** every neighbouring pair, **choose** the most frequent, **merge** it everywhere - so we build it in six small, runnable parts. Run each cell in order; every part uses what the one before it defined.

### Part 1 - Start with one token per character

`list(text)` splits the string into individual characters - the smallest possible tokens. The blank space is a token too, which is why you see `' '` between words. Right now the tokenizer has learned nothing: one character = one token.

**You'll see:** the first dozen character-tokens, spaces included.

In [ ]:
text = "low lower lowest new newer newest"

# list(text) separates the string character by character.
tokens = list(text)

print(tokens[:12])
# ['l', 'o', 'w', ' ', 'l', 'o', 'w', 'e', 'r', ' ', 'l', 'o']

### Part 2 - Count neighbouring pairs

`get_pairs` walks the token list with `zip(tokens, tokens[1:])` and counts every adjacent pair in a `Counter`. "Which two pieces sit next to each other most often?" is the entire signal BPE uses to decide what to merge.

**You'll see:** the five most common pairs - the winner is `('w', 'e')` at 4.

In [ ]:
from collections import Counter


def get_pairs(tokens):
    """Return how often every adjacent pair appears."""
    pairs = Counter()

    # zip creates: (token 0, token 1), (token 1, token 2), ...
    for left, right in zip(tokens, tokens[1:]):
        pairs[(left, right)] += 1

    return pairs


pairs = get_pairs(tokens)
print(pairs.most_common(5))
# [(('w', 'e'), 4), (('l', 'o'), 3), ...]

### Part 3 - Merge one selected pair

`merge_pair` rebuilds the list, gluing **every** occurrence of one chosen pair into a single token. The key line is `i += 2`: when a pair is merged, two old tokens became one, so the pointer jumps forward by two; otherwise it moves by one.

**You'll see:** `['l','o','w','l','o','w']` become `['lo','w','lo','w']`.

In [ ]:
def merge_pair(tokens, pair):
    """Merge every occurrence of one selected pair."""
    result = []
    i = 0

    while i < len(tokens):
        has_next_token = i + 1 < len(tokens)
        current_pair = (tokens[i], tokens[i + 1]) if has_next_token else None

        if current_pair == pair:
            result.append(tokens[i] + tokens[i + 1])
            i += 2              # Two old tokens were consumed.
        else:
            result.append(tokens[i])
            i += 1              # Only one token was consumed.

    return result


example = ["l", "o", "w", "l", "o", "w"]
print(merge_pair(example, ("l", "o")))
# ['lo', 'w', 'lo', 'w']

### Part 4 - Repeat: count, choose, merge

Read the training loop as one sentence: **count the pairs -> take the most frequent -> merge it -> save the rule -> repeat.** `most_common(1)[0]` returns the winning pair and its count; `learned_merges` keeps the rules in order (order matters when tokenizing); `vocab` gives each final token an integer id. This cell only defines `train_bpe` - Part 6 runs it.

In [ ]:
def train_bpe(text, num_merges=8):
    tokens = list(text)
    learned_merges = []

    for step in range(num_merges):
        # 1. Count all neighbouring pairs.
        pairs = get_pairs(tokens)
        if not pairs:
            break

        # 2. Select the pair with the highest count.
        best_pair, count = pairs.most_common(1)[0]

        # 3. Merge it everywhere and remember the rule.
        tokens = merge_pair(tokens, best_pair)
        learned_merges.append(best_pair)

        print(
            f"Step {step + 1}: {best_pair} appeared {count} times "
            f"-> {len(tokens)} tokens remain"
        )

    # Give every final token a numeric ID.
    vocab = {
        token: token_id
        for token_id, token in enumerate(sorted(set(tokens)))
    }

    return tokens, learned_merges, vocab

### Part 5 - Apply the learned rules to new text

Training and tokenizing are different jobs. `tokenize` does **not** re-learn anything - it just replays the merge rules in the same order they were learned, so an unseen word breaks into the same subwords the training corpus produced.

In [ ]:
def tokenize(text, learned_merges):
    tokens = list(text)

    # Replay the merge rules in the same order used during training.
    for pair in learned_merges:
        tokens = merge_pair(tokens, pair)

    return tokens

### Part 6 - Run the complete tokenizer

Train on the sample text, then tokenize two new words. A frequent pattern collapses into a bigger token; a rare pattern stays split.

**You'll see:** eight merge steps, then `lower  -> ['lo', 'we', 'r']` (5 letters -> 3 tokens) and `newest -> ['n', 'e', 'we', 'st']`.

In [ ]:
training_text = "low lower lowest new newer newest"

final_tokens, learned_merges, vocab = train_bpe(
    training_text,
    num_merges=8,
)

print("\nLearned merges:", learned_merges)
print("Vocabulary:", vocab)
print("lower  ->", tokenize("lower", learned_merges))
print("newest ->", tokenize("newest", learned_merges))

# Expected final tests:
# lower  -> ['lo', 'we', 'r']
# newest -> ['n', 'e', 'we', 'st']

### All six parts in one place

Here is the whole tokenizer, top to bottom - the exact same code you just built in six steps. Run it and you get the same result. Not so big after all.

In [ ]:
from collections import Counter


def get_pairs(tokens):
    """Return how often every adjacent pair appears."""
    pairs = Counter()
    for left, right in zip(tokens, tokens[1:]):
        pairs[(left, right)] += 1
    return pairs


def merge_pair(tokens, pair):
    """Merge every occurrence of one selected pair."""
    result = []
    i = 0

    while i < len(tokens):
        has_next_token = i + 1 < len(tokens)
        current_pair = (tokens[i], tokens[i + 1]) if has_next_token else None

        if current_pair == pair:
            result.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            result.append(tokens[i])
            i += 1

    return result


def train_bpe(text, num_merges=8):
    tokens = list(text)
    learned_merges = []

    for step in range(num_merges):
        pairs = get_pairs(tokens)
        if not pairs:
            break

        best_pair, count = pairs.most_common(1)[0]
        tokens = merge_pair(tokens, best_pair)
        learned_merges.append(best_pair)

        print(
            f"Step {step + 1}: {best_pair} appeared {count} times "
            f"-> {len(tokens)} tokens remain"
        )

    vocab = {
        token: token_id
        for token_id, token in enumerate(sorted(set(tokens)))
    }
    return tokens, learned_merges, vocab


def tokenize(text, learned_merges):
    tokens = list(text)
    for pair in learned_merges:
        tokens = merge_pair(tokens, pair)
    return tokens


training_text = "low lower lowest new newer newest"
final_tokens, learned_merges, vocab = train_bpe(training_text, num_merges=8)

print("\nFinal training tokens:", final_tokens)
print("Learned merges:", learned_merges)
print("Vocabulary:", vocab)
print("lower  ->", tokenize("lower", learned_merges))
print("newest ->", tokenize("newest", learned_merges))

## 2 - Compare real tokenizers across models and languages

The same sentence costs a different number of tokens depending on the tokenizer. Here you run GPT-4's `cl100k_base` and GPT-4o's `o200k_base` over English, Hindi, Telugu, code, JSON and emoji, and count the tokens each produces. This is where the "multilingual token tax" gets concrete: non-Latin scripts often cost several times more tokens for the same meaning - so the same request costs more and eats more of the context window.

In [ ]:
# =============================================
# COMPARE TOKENIZERS ACROSS MODELS
# Same text → different token counts.
# pip install tiktoken
# =============================================

import tiktoken

# GPT-4's tokenizer
enc_gpt4 = tiktoken.get_encoding("cl100k_base")     # GPT-4, GPT-3.5
enc_gpt4o = tiktoken.get_encoding("o200k_base")     # GPT-4o

test_texts = [
    ("English",  "Tokenization is the first step in building an LLM."),
    ("Hindi",    "टोकनाइज़ेशन एलएलएम बनाने का पहला कदम है।"),
    ("Telugu",   "టోకెనైజేషన్ LLM నిర్మించడంలో మొదటి దశ."),
    ("Code",     "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)"),
    ("JSON",     '{"name": "Netsetos", "students": 10000, "rating": 4.9}'),
    ("Emoji",    "I love AI! 🤖🚀💡 Let's build something amazing!"),
]

print("Token count comparison:\n")
print(f"  {'Language':10s} {'GPT-4':>7s} {'GPT-4o':>8s} {'Diff':>6s}  Text")
print(f"  {'─'*75}")

for lang, text in test_texts:
    gpt4_tokens = enc_gpt4.encode(text)
    gpt4o_tokens = enc_gpt4o.encode(text)
    diff = len(gpt4_tokens) - len(gpt4o_tokens)
    diff_str = f"-{abs(diff)}" if diff > 0 else f"+{abs(diff)}" if diff < 0 else "="
    print(f"  {lang:10s} {len(gpt4_tokens):>5d}   {len(gpt4o_tokens):>5d}   {diff_str:>5s}  {text[:45]}")

# Show actual tokens for English
print(f"\nGPT-4 tokens for English:")
tokens = enc_gpt4.encode(test_texts[0][1])
decoded = [enc_gpt4.decode([t]) for t in tokens]
print(f"  {decoded}")

print(f"\nGPT-4 tokens for Hindi:")
tokens = enc_gpt4.encode(test_texts[1][1])
print(f"  Token count: {len(tokens)} (vs {len(enc_gpt4.encode(test_texts[0][1]))} for English)")
ratio = len(tokens) / len(enc_gpt4.encode(test_texts[0][1]))
print(f"  Hindi uses ~{ratio:.1f}x more tokens → costs ~{ratio:.1f}x more per prompt (cl100k)!")

# Output:
#   Language     GPT-4   GPT-4o   Diff
#   English         12       12      =
#   Hindi           45       15    -30   <- o200k is far better on Devanagari
#   Telugu          63       18    -45
#   Code            24       24      =
#   JSON            23       23      =
#   Emoji           18       15     -3
# Newer tokenizers (o200k) slash the multilingual token tax.

**Explanation**

This is a measurement harness, not a model call: encode the same strings with two different tokenizers and compare the lengths. `encode` returns token ids, `len` counts them, and `decode` reverses one id at a time so you can inspect the actual pieces. The ratio at the end turns the multilingual penalty into a single number.

**How the code works - step by step**
- Loads two real tokenizers with `tiktoken.get_encoding`: `cl100k_base` (GPT-4 / 3.5) and `o200k_base` (GPT-4o).
- `test_texts` - six samples spanning languages and formats.
- For each sample, `enc.encode(text)` returns the list of token ids and `len(...)` is the token count; the loop prints the GPT-4 vs GPT-4o counts and their difference.
- The English block then `decode`s each token id back to its string, so you can *see* the exact pieces the tokenizer chose.
- The Hindi block computes the ratio of Hindi to English tokens - the token tax as a single number.

**What you'll see:** English is ~12 tokens on both tokenizers; Hindi / Telugu are far cheaper on `o200k` than `cl100k` (e.g. 45 -> 15), showing newer tokenizers cut the multilingual tax. More tokens = more cost and less usable context.

## 3 - Price a prompt to the rupee

Tokens are the unit you pay for, so prompt wording is a direct cost lever. This cell counts the tokens in a tight prompt versus a chatty one carrying the *same* instruction, then multiplies by real 2026 input prices across Gemini, GPT-5 and Claude to show the monthly bill at 10,000 calls/day.

In [ ]:
# =============================================
# TOKEN ECONOMICS CALCULATOR
# How much does your prompt actually cost?
# =============================================

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Pricing per 1M tokens (USD), as of 2026 - always check current rates
PRICES = {
    "Gemini 2.5 Flash (input)":   0.30,
    "Gemini 2.5 Flash (output)":  2.50,
    "GPT-5 (input)":              1.25,
    "GPT-5 (output)":            10.00,
    "Claude Sonnet 4.6 (input)":   3.00,
    "Claude Sonnet 4.6 (output)": 15.00,
}

def analyze_cost(text: str, label: str, daily_calls: int = 10000):
    tokens = enc.encode(text)
    n = len(tokens)
    print(f"\n  📊 {label}")
    print(f"  Characters: {len(text):,} | Tokens: {n:,} | Ratio: {len(text)/n:.1f} chars/token")
    print(f"  Cost per call ({daily_calls:,}/day):")
    for model, price in PRICES.items():
        cost_per_call = n * price / 1e6
        daily = cost_per_call * daily_calls
        monthly = daily * 30
        if "input" in model:
            print(f"    {model:30s}: ${cost_per_call:.6f}/call → ₹{monthly*84:.0f}/month")

print("Token Economics:\n")

# Compare: short prompt vs verbose prompt
short = "Summarize: key points only, 3 bullets"
verbose = "I would like you to please kindly summarize the following text. It is important that you provide the key points in a bulleted list format with exactly 3 bullets."

analyze_cost(short, "Short prompt (optimized)")
analyze_cost(verbose, "Verbose prompt (wasteful)")

print(f"\n  💡 Same instruction. Verbose = {len(enc.encode(verbose))/len(enc.encode(short)):.1f}x more tokens → {len(enc.encode(verbose))/len(enc.encode(short)):.1f}x more cost!")

# Output:
# Short prompt (optimized):  11 tokens
#   Gemini 2.5 Flash (input)  : $0.000003/call -> ~Rs 83/month   (10k calls/day)
#   GPT-5 (input)             : $0.000014/call -> ~Rs 346/month
#   Claude Sonnet 4.6 (input) : $0.000033/call -> ~Rs 832/month
# Verbose prompt (wasteful): 33 tokens -> 3.0x more tokens = 3.0x more cost

**Explanation**

This is a cost model, not an API call. It counts the tokens in a prompt and multiplies by published per-token prices to project a monthly bill, then runs the same instruction in a lean vs a padded form so the only variable is wording. The point is the multiplier between them, not the absolute rupee figure.

**How the code works - step by step**
- `PRICES` - per-million-token USD rates (input and output) for three frontier models, as of 2026.
- `analyze_cost(text, label, daily_calls)` - encodes the text to count tokens, then for each **input** price computes cost per call, scales it to a monthly figure, and converts USD -> INR (x84).
- It runs on two prompts with identical intent: `short` (optimized) vs `verbose` (padded with politeness).
- The last line divides verbose tokens by short tokens to show the multiplier.

**What you'll see:** the short prompt is ~11 tokens, the verbose one ~33 - **3x the tokens, so 3x the cost** for the same meaning. Trimming wording is free money at scale.

## 4 - From tokens to meaning: embeddings

Token ids are just labels; **embeddings** turn text into vectors where distance means similarity. We do this in two steps: first build the whole idea by hand with NumPy (no API key, no network), then call a real embedding model and watch the same thing happen at 768 dimensions. This is the exact mechanism behind RAG search in Module 8.

### First, by hand - no API key needed

An embedding is just a list of numbers that places a word somewhere in "meaning-space". Here we hand-place four words on a tiny 2D map - one axis for royalty, one axis for food - and measure closeness with cosine similarity. This is exactly what `gemini-embedding-001` does, only by hand and in 2 dimensions instead of 3072.

In [ ]:
# =============================================
# EMBEDDINGS FROM SCRATCH - meaning as coordinates
# No API key needed here - just numpy. This is the same
# idea gemini-embedding-001 does, only by hand and in 2D.
# =============================================
import numpy as np

# Hand-made 2D "meaning" vectors: axis 0 = royalty, axis 1 = food
vectors = {
    "king":    np.array([0.95, 0.10]),
    "queen":   np.array([0.90, 0.15]),
    "biryani": np.array([0.10, 0.95]),
    "dosa":    np.array([0.05, 0.90]),
}

def cosine(a, b):
    # closeness = the angle between two vectors (length does not matter)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for word in ["queen", "biryani", "dosa"]:
    score = cosine(vectors["king"], vectors[word])
    print(f"king vs {word:8s} -> {score:.2f}")

# Output:
# king vs queen    -> 1.00   (both point 'royalty' -> same direction)
# king vs biryani  -> 0.21   (royalty vs food -> nearly perpendicular)
# king vs dosa     -> 0.16

**Explanation**

This cell builds an embedding out of nothing but NumPy, so you can see the mechanism before any API is involved. The vectors are typed by hand; the only real machinery is `cosine`, which is the same similarity function used everywhere in retrieval.

**How the code works - step by step**
- `vectors` - four words, each given a 2-number "address": axis 0 is royalty, axis 1 is food. `king` is `[0.95, 0.10]` (very royal, not food); `biryani` is `[0.10, 0.95]` (the reverse).
- `cosine(a, b)` - the dot product divided by the product of the two lengths. That is the **angle** between the vectors, so only direction matters, not magnitude.
- The loop compares `king` against the other three words and prints each score to 2 decimals.

**What you'll see:** `king vs queen -> 1.00` (both point the royalty direction), `king vs biryani -> 0.21` and `king vs dosa -> 0.16` (royalty vs food is close to perpendicular). Meaning has become geometry - and that is the whole trick behind embedding search.

### Now the real thing - gemini-embedding-001

Same idea, but the numbers come from a trained model instead of your hand: 3072 dimensions (sliced to 768 here) and it already knows `king` and `queen` are related because it read the whole internet. This cell embeds six words and builds a full cosine-similarity matrix, so you can watch related words score high and unrelated ones score low.

> **Needs a Gemini API key.** Set `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) in your environment before running this cell.

In [ ]:
# =============================================
# EMBEDDINGS - From tokens to meaning vectors
# pip install google-genai numpy   (google-genai is the current SDK;
# the old google-generativeai package is deprecated)
# =============================================

from google import genai
from google.genai import types
import numpy as np
import os

# Client reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

def get_embedding(text: str) -> np.ndarray:
    # gemini-embedding-001: 3072 dims by default; 768 is plenty here
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    return np.array(result.embeddings[0].values)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Get embeddings for related and unrelated words
words = ["king", "queen", "prince", "biryani", "programming", "Python"]
embeddings = {w: get_embedding(w) for w in words}

print("Embedding Similarity Matrix:\n")
print(f"  {'':12s}" + "".join(f"{w:>12s}" for w in words))
for w1 in words:
    row = f"  {w1:12s}"
    for w2 in words:
        sim = cosine_sim(embeddings[w1], embeddings[w2])
        row += f"{sim:12.3f}"
    print(row)

print(f"\n  Embedding dimension: {len(embeddings['king'])}")
print(f"\n  Key observations:")
print(f"  • king ↔ queen:       HIGH (both royalty)")
print(f"  • king ↔ biryani:     LOW (unrelated)")
print(f"  • programming ↔ Python: HIGH (related concepts)")
print(f"  • biryani ↔ Python:   LOW (food vs code)")
print(f"\n  This is how RAG search works (Module 8):")
print(f"  convert query → embedding → find nearest documents.")

# Output: (illustrative - exact cosine values vary by model version)
#               king  queen  prince biryani  prog. Python
#   king        1.00   0.78   0.72   0.31   0.30   0.33
#   queen       0.78   1.00   0.70   0.30   0.29   0.31
#   biryani     0.31   0.30   0.29   1.00   0.28   0.27
#   Embedding dimension: 768
#   king <-> queen: HIGH | king <-> biryani: LOW | programming <-> Python: HIGH

**Explanation**

This cell turns words into vectors and measures the angles between them. `get_embedding` is one API call per word; `cosine_sim` compares any two vectors; the rest is bookkeeping to print a full similarity grid. The grid - not the individual numbers - is the payoff: meaning shows up as high similarity between related words.

**How the code works - step by step**
- `client = genai.Client()` - reads the API key from the environment.
- `get_embedding(text)` - calls `embed_content` on `gemini-embedding-001` with `output_dimensionality=768` (a Matryoshka slice of the full 3072 dims) and returns the vector as a NumPy array.
- `cosine_sim(a, b)` - cosine similarity: the dot product divided by the product of the norms, i.e. the angle between two vectors (1 = same direction, 0 = unrelated).
- Embeds six words, then prints the full pairwise similarity matrix.

**What you'll see:** a 6x6 matrix where `king<->queen` and `programming<->Python` are high (~0.7-0.8) while cross-topic pairs like `biryani<->Python` are low (~0.3); the dimension prints as 768. The same "embed it, compare by cosine" idea powers "embed the query, find the nearest documents."

## 5 - Why LLMs miscount letters: tokenization quirks

Several famous LLM failures are direct side-effects of tokenization: the model never sees individual letters, only merged chunks. We run four quick probes with GPT-4o's tokenizer, one at a time, then all together at the end - no model reasoning involved, only the tokenizer's splits.

### Quirk 1 - The strawberry problem

The model reads `strawberry` as a few chunks, not 10 letters, so it cannot reliably count the r's.

**You'll see:** `['st', 'raw', 'berry']` (3 chunks), and the real r-count next to the 2 that models often guess.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

# ── 1. The Strawberry Problem ──
word = "strawberry"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'strawberry' → {decoded}")
print(f"Model sees {len(tokens)} chunks, NOT 10 letters")
print(f"Actual r count: {word.count('r')} - model often says 2\n")

### Quirk 2 - Reversing letters

`hello` is a single token. The letters are not individually visible, which is why flipping them is surprisingly hard for a model.

In [ ]:
# ── 2. String Reversal ──
word = "hello"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'hello' → {decoded}")
print(f"Model can't easily reverse within merged tokens\n")

### Quirk 3 - Why LLM arithmetic is shaky

The same digits get chunked differently depending on the number, so the model never sees numbers in one consistent form.

In [ ]:
# ── 3. Arithmetic Inconsistency ──
for num in ["234", "2345", "23456", "1000000"]:
    tokens = enc.encode(num)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  '{num}' → {decoded} ({len(tokens)} tokens)")
print(f"Numbers tokenize inconsistently - that's why LLM math fails\n")

### Quirk 4 - The multilingual token tax

The same sentence costs about 2-3x more tokens in Hindi or Telugu than English, so it costs about 2-3x more per API call.

In [ ]:
# ── 4. The Multilingual "Token Tax" ──
texts = {
    "English":  "AI is transforming how we learn and work.",
    "Hindi":    "कृत्रिम बुद्धिमत्ता हमारे सीखने और काम करने के तरीके को बदल रही है।",
    "Telugu":   "కృత్రిమ మేధస్సు మనం నేర్చుకునే విధానాన్ని మారుస్తోంది.",
}
en_count = len(enc.encode(texts["English"]))
for lang, text in texts.items():
    count = len(enc.encode(text))
    ratio = count / en_count
    print(f"  {lang:<10s} {count:>3} tokens ({ratio:.1f}x English)")
print(f"\n  Hindi costs ~{len(enc.encode(texts['Hindi']))/en_count:.1f}x more → same meaning, higher API bill!")
print(f"  Token Tax (arXiv 2509.05486): Arabic/Hindi/Telugu run ~3-5x English")

### All four probes in one place

Here are the four quirks in a single block - the exact code from above, together. It is just four tiny encode-and-print checks.

In [ ]:
# =============================================
# TOKENIZATION QUIRKS - Why LLMs struggle with
# letter counting, reversal, and arithmetic
# =============================================

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

# ── 1. The Strawberry Problem ──
word = "strawberry"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'strawberry' → {decoded}")
print(f"Model sees {len(tokens)} chunks, NOT 10 letters")
print(f"Actual r count: {word.count('r')} - model often says 2\n")

# ── 2. String Reversal ──
word = "hello"
tokens = enc.encode(word)
decoded = [enc.decode([t]) for t in tokens]
print(f"'hello' → {decoded}")
print(f"Model can't easily reverse within merged tokens\n")

# ── 3. Arithmetic Inconsistency ──
for num in ["234", "2345", "23456", "1000000"]:
    tokens = enc.encode(num)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  '{num}' → {decoded} ({len(tokens)} tokens)")
print(f"Numbers tokenize inconsistently - that's why LLM math fails\n")

# ── 4. The Multilingual "Token Tax" ──
texts = {
    "English":  "AI is transforming how we learn and work.",
    "Hindi":    "कृत्रिम बुद्धिमत्ता हमारे सीखने और काम करने के तरीके को बदल रही है।",
    "Telugu":   "కృత్రిమ మేధస్సు మనం నేర్చుకునే విధానాన్ని మారుస్తోంది.",
}
en_count = len(enc.encode(texts["English"]))
for lang, text in texts.items():
    count = len(enc.encode(text))
    ratio = count / en_count
    print(f"  {lang:<10s} {count:>3} tokens ({ratio:.1f}x English)")
print(f"\n  Hindi costs ~{len(enc.encode(texts['Hindi']))/en_count:.1f}x more → same meaning, higher API bill!")
print(f"  Token Tax (arXiv 2509.05486): Arabic/Hindi/Telugu run ~3-5x English")

# Output:
# 'strawberry' -> ['st', 'raw', 'berry']   (3 chunks, NOT 10 letters)
# Actual r count: 3   (the model often says 2)
# 'hello' -> ['hello']
# '234' -> ['234'] (1); '2345' -> ['234','5'] (2); '1000000' -> ['100','000','0'] (3)
# English 1.0x | Hindi 2.2x | Telugu 2.4x English (o200k)

## 6 - Special tokens and chat templates

Beyond ordinary text, every model reserves a handful of **control tokens** - start / end of sequence, padding, role markers - and wraps chat turns in a model-specific template. Getting these wrong is a top cause of broken fine-tunes and runaway generation. This cell lists the common special tokens and the chat formats for OpenAI, Llama 3 and Mistral.

In [ ]:
# =============================================
# SPECIAL TOKENS - The hidden control signals
# =============================================

# Every model has reserved token IDs with special roles:
special_tokens = {
    "[BOS] / <s>":         "Beginning of sequence - signals start",
    "[EOS] / </s>":        "End of sequence - model stops generating",
    "[PAD]":               "Padding - fills short sequences in batches",
    "[UNK]":               "Unknown - fallback for missing vocab",
    "<|endoftext|>":       "OpenAI's document separator",
    "<|im_start|>":        "Chat message boundary (OpenAI)",
    "[INST] / [/INST]":    "Instruction boundaries (Mistral/Llama 2)",
    "<|start_header_id|>": "Role marker (Llama 3)",
}

print("SPECIAL TOKENS ACROSS MODELS:\n")
for token, role in special_tokens.items():
    print(f"  {token:<25s} {role}")

# Chat template example - how models know who's talking
print(f"""
CHAT TEMPLATES - Different models, different formats:

OpenAI (GPT-4):
  <|im_start|>system
  You are a helpful assistant.<|im_end|>
  <|im_start|>user
  What is GenAI?<|im_end|>

Llama 3:
  <|begin_of_text|><|start_header_id|>system<|end_header_id|>
  You are a helpful assistant.<|eot_id|>

Mistral:
  [INST] What is GenAI? [/INST]

Why this matters:
  - Wrong template during fine-tuning = broken model
  - Missing EOS token = model generates forever
  - Double BOS tokens = confused predictions
  - Module 9 (fine-tuning) requires getting this exactly right
""")

# Output:
# SPECIAL TOKENS ACROSS MODELS:
#   [BOS] / <s>            Beginning of sequence - signals start
#   [EOS] / </s>           End of sequence - model stops generating
#   <|endoftext|>          OpenAI's document separator
#   <|im_start|>           Chat message boundary (OpenAI)
#   [INST] / [/INST]       Instruction boundaries (Mistral/Llama 2)
#   <|start_header_id|>    Role marker (Llama 3)
# ...followed by the OpenAI / Llama 3 / Mistral chat-template formats.

**Explanation**

This is a reference cell: two printed lookups, no computation. The first lists reserved control tokens and their jobs; the second shows how three model families wrap the same chat turn. You'll come back to it whenever a fine-tune or a chat format misbehaves.

**How the code works - step by step**
- `special_tokens` - a reference map of the reserved tokens (`[BOS]/[EOS]/[PAD]/[UNK]`, `<|endoftext|>`, `<|im_start|>`, `[INST]`, Llama-3 header markers) and what each one signals.
- Prints them as an aligned table.
- The multi-line block then shows how the *same* conversation is framed differently by OpenAI, Llama 3 and Mistral, and lists the failure modes: wrong template -> broken fine-tune, missing EOS -> generates forever, double BOS -> confused predictions.

**What you'll see:** the special-token table and three chat-template layouts side by side. No model call here - it's a reference you'll rely on in Module 9 (fine-tuning), where the template has to match exactly.

## 7 - The 2026 embedding landscape and dimension trade-offs

Choosing an embedding model is a cost / quality / dimension decision. This cell tabulates the main 2026 options (dimensions, price, provider) and shows how vector dimension drives storage cost - plus **Matryoshka** embeddings, which let you train once and truncate to a smaller size with minimal quality loss.

In [ ]:
# =============================================
# EMBEDDING LANDSCAPE - Models, Dimensions, RAG
# =============================================

# 2026 embedding models (MTEB leaderboard, approximate - check current rates)
models = [
    ("gemini-embedding-001",   3072,  "~$0.15/M", "Google",   "~#1 English MTEB; Matryoshka 3072/1536/768"),
    ("Qwen3-Embedding-8B",     4096,  "open",     "Alibaba",  "~#1 multilingual MTEB; self-host, instruct"),
    ("text-embedding-3-large", 3072,  "$0.13/M",  "OpenAI",   "Matryoshka: truncate 256-3072"),
    ("text-embedding-3-small", 1536,  "$0.02/M",  "OpenAI",   "Best value for most RAG apps"),
    ("voyage-3.5",             1024,  "~$0.06/M", "Voyage",   "Strong retrieval, good price/quality"),
    ("BGE-M3",                1024,  "open",     "BAAI",     "Open-source, multilingual, long-context"),
]

print(f"{'Model':<26s} {'Dims':>5} {'Cost':>10} {'Provider':<10} Notes")
print("-" * 90)
for name, dims, cost, provider, notes in models:
    print(f"  {name:<24s} {dims:>5} {cost:>10} {provider:<10} {notes}")

# Dimension vs Storage Cost
print(f"""
DIMENSION TRADE-OFFS (10M vectors, approximate):
  256 dims  → ~10 GB  → ~$1.25/month → ~95% of max accuracy
  768 dims  → ~30 GB  → ~$3.75/month → ~98% of max accuracy
  3072 dims → ~120 GB → ~$15/month   → 100% accuracy

Matryoshka embeddings (gemini-embedding-001, text-embedding-3):
  Train once at full size, truncate to ANY size at query time.
  A 768-dim slice keeps ~98% of the quality of the full 3072 dims.
  → Use 768 for most RAG apps, save ~75% storage vs 3072.

For Module 8 (RAG): embedding quality = retrieval quality = answer quality.
  Sensible defaults: gemini-embedding-001 (768) or text-embedding-3-small.
""")

# Output:
# Model                   Dims      Cost  Provider  Notes
# gemini-embedding-001    3072  ~$0.15/M  Google    ~#1 English MTEB; Matryoshka
# Qwen3-Embedding-8B      4096      open  Alibaba   ~#1 multilingual MTEB; self-host
# text-embedding-3-large  3072   $0.13/M  OpenAI    Matryoshka 256-3072
# text-embedding-3-small  1536   $0.02/M  OpenAI    Best value for RAG
# voyage-3.5              1024  ~$0.06/M  Voyage    Strong retrieval
# BGE-M3                  1024      open  BAAI      Open-source, multilingual

**Explanation**

A decision-support cell: it prints a comparison table and the storage math behind embedding dimension, then explains Matryoshka truncation. There is nothing to run against an API - it's the cheat-sheet for choosing an embedding model in Module 8.

**How the code works - step by step**
- `models` - a comparison table of 2026 embedding models: dimensions, rough price, provider, and a one-line note (MTEB rank, Matryoshka support, open vs hosted).
- Prints it aligned, then a second block lays out the dimension -> storage -> accuracy trade-off for 10M vectors (256 vs 768 vs 3072 dims).
- Explains Matryoshka: a 768-dim slice of a 3072-dim model keeps ~98% of the quality at ~1/4 the storage.

**What you'll see:** the model table and the storage math, landing on sensible RAG defaults (`gemini-embedding-001` at 768, or `text-embedding-3-small`). Reference for Module 8; no API call.

## Recap

You built tokenization and embeddings end to end: a from-scratch BPE tokenizer, real-tokenizer comparisons, prompt-cost math, live embeddings with cosine similarity, the quirks tokenization causes, and how to pick an embedding model. Tokens set your bill and context budget; embeddings turn text into searchable meaning - the foundation for retrieval in Module 8 and fine-tuning in Module 9.